# 06 · Verification and evaluation

**Session 01 · Building AI applications for research and data · notebook 6 of 6**

## Objective

Notebooks 01 to 05 put the right evidence in front of the model: context, retrieved passages, exact records from a data tool, and a bounded agent loop. The answer still contains numbers in plain text, and nothing in the application checks them against the records. This notebook adds the check, then measures the whole application.

Verification follows the Proof-Carrying Numbers protocol (PCN), the program's approach to trustworthy numbers ([paper](https://arxiv.org/abs/2509.06902), [code](https://github.com/worldbank/pcn)). The model writes every figure from the data tool inside a claim tag that names the record the figure came from and the rule for comparing the two. A verifier reads each tag, fetches the record, compares, and marks the number verified or flagged. A number without a tag is unmarked. Evaluation runs a fixed set of eight questions with known evidence through two configurations of the application and compares the results in one table.

Four terms:

- A **claim tag** is a label the model writes around a number. It names the record the number comes from and how to compare the two. It is the `claim_id` of notebook 05 with a comparison rule added.
- A **verifier** reads the tags and checks each number against its record. Here it is `pcn-core`, the Python core of the protocol. Each number gets one mark: **verified** (it matches), **flagged** (it does not), or **unmarked** (no tag, so nothing to check against).
- An **evaluation suite** is a fixed list of questions, each with the evidence and the behaviour a correct answer needs.
- A **configuration** is one version of the application. The suite compares two.

The session has used four kinds of check:

| Check | Question it answers | Where |
|---|---|---|
| Retrieval | Did the application find the right evidence? | Notebook 03, section 6 |
| Grounding | Is the answer written from that evidence? | Citations in the answer |
| Verification | Does each number match its source record? | Claim tags and `pcn-core`, this notebook |
| Evaluation | Does the application behave across many questions? | The suite, this notebook |

## The application after this notebook

![Diagram: the question goes to the agent with tools of notebook 05, which writes an answer with claim tags; the verifier, pcn-core, puts a mark on each number: verified, flagged or unmarked. Separately, the evaluation suite of eight questions produces a table per question for each configuration of the application.]()

## Contents

1. [Section 1](#scrollTo=section-1) shows a claim tag and the main question answered with every tagged number checked: 14 verified, none flagged, and 21 untagged numbers left unmarked.
2. [Section 2](#scrollTo=section-2) replays an answer with one altered figure and one untagged figure, and shows the flag and the absence of a mark.
3. [Section 3](#scrollTo=section-3) defines two configurations of the application: retrieval only, and tools with claims.
4. [Section 4](#scrollTo=section-4) lists the eight evaluation questions and runs both configurations through them.
5. [Section 5](#scrollTo=section-5) reads the table. Each configuration fails one question the other passes, and no single column shows it.
6. [Section 6](#scrollTo=section-6) states what the suite does not cover.

## How to run it

Run the cells in order. Without an API key, the model's answers are recorded ones, labelled `[recorded output: ...]`, as in notebook 01. With a key in `WORKSHOP_API_KEY`, the same cells call the model live, the answer streams into its bubble as the model writes, and the wording differs from the recorded answers. The checks and the tables are computed when a cell runs, in both modes. The exercises at the end run without a key.

The next cell installs the session's tools and loads its helper module. The first run takes about a minute. The cell is setup only.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # the program's repository
BRANCH = "main"   # the branch the Colab setup clones
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


The helper functions used in this notebook, in the order they appear. `w.use` and `w.show` are as in notebook 01.

| Call | What it does |
|---|---|
| `w.make_search_tool(store, k, since)`, `w.get_indicator` | The two tools of notebooks 04 and 05: a search of the store with a date filter, and a lookup of a World Development Indicators series. |
| `w.indicator_lookup(...)` | The function behind `get_indicator`, called directly in section 1. |
| `w.observations_from_results(results)` | A dictionary from claim id to the value the tool returned. |
| `w.show_chat(question, answer, observations, results, raw)` | The answer as a chatbot draws it, with a mark after each checked number. |
| `w.verify(answer, observations)` | The verifier: one check per number in the answer, with its status. |
| `w.show_checks(checks)` | The checks as a table. |
| `w.tool_results(result, name)`, `w.final_text(result)` | The parsed tool results and the final answer of an agent run. |
| `w.grounded(answer, passages)` | Whether the answer cites one of the retrieved passages. |
| `w.EvalRow`, `w.show_eval(rows)` | One row of the evaluation table, and the table. |

The next cell rebuilds the store, creates the two tools of notebooks 04 and 05, and writes the instructions. The new rule is the claim tag. Its `id` is `indicator:country:year`, the key under which the tool returned the value. `policy="rounded"` with `decimals="1"` means that the record, rounded to one decimal, must equal the number shown; `policy="exact"` requires equality.

In [2]:
import json
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

docs = w.load_documents()
store = w.build_store(w.split_sections(docs))   # the store of notebooks 02 and 03
search_documents = w.make_search_tool(store, k=3, since="2020")   # the tools of notebooks 04 and 05
tools = [search_documents, w.get_indicator]

INSTRUCTIONS_CLAIMS = """You are a research assistant for development data.
Use get_indicator for every statistic and search_documents for definitions and context. Do not quote figures from memory.
Write every number that comes from get_indicator inside a claim tag that names its claim_id, like
<claim id="IT.NET.USER.ZS:KEN:2024" policy="rounded" decimals="1">35.0</claim>.
Name the indicator and the source. If the tools do not provide what is needed, say so.
If the question is ambiguous, ask which reading is meant."""

## 1. Claim tags

A claim tag is a label around a number. The verifier takes the record named by the tag's id, applies the tag's rule, and compares the result with the number shown. The animation shows the three outcomes: a tagged figure that matches its record is verified, a tagged figure that differs is flagged, and a figure without a tag stays unmarked.

![Animation: the model writes 35.0 inside a claim tag that names the record IT.NET.USER.ZS:KEN:2024 and the rule rounded to one decimal; the verifier fetches the record, 34.9762, rounds it to 35.0 and marks the number verified. A second figure, 35.7 for Rwanda, differs from its record, 31.7033, and is flagged. A third figure, about 70 percent, has no tag and stays unmarked.]()

The next cell gives the model the four series the main question needs, placed in the prompt as notebook 04 did, and asks it to answer with tags. `w.observations_from_results` builds the dictionary from claim id to the value the tool returned, which is what the verifier compares against. `w.show_chat` draws the answer as a chatbot would: a mark after each checked number, a tooltip on hover with the record and the rule, and, with `raw=True`, a fold that holds the tagged text as the model wrote it.

In [3]:
results = [w.indicator_lookup(i, c, start=2010) for i in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"] for c in ["Kenya", "Rwanda"]]
observations = w.observations_from_results(results)   # claim_id -> the value the tool returned

w.use("nb06_claims_main")
reply = model.invoke([SystemMessage(content=INSTRUCTIONS_CLAIMS), HumanMessage(content="DATA:\n" + json.dumps(results) + "\n\nQUESTION: " + w.MAIN_QUESTION)])
w.show_chat(w.MAIN_QUESTION, reply.content, observations, results, raw=True)

Year,Kenya,Rwanda
2010,7.2✓,8.0✓
2015,16.6✓,12.5✓
2020,24.7✓,23.8✓
2024,35.0✓,31.7✓
Year,Kenya,Rwanda
2010,1092✓,587✓
2020,1928✓,803✓
2024,2133✓,1060✓


The model put its figures in two tables, one tag per cell, and wrote four patterns under them. Every tagged number carries a ✓. The numbers in the patterns carry no mark: they are the same figures repeated without tags, or values the model computed, such as a gain of 27.8 points or a ratio of 1.95. The next cell lists every check as a table.

In [4]:
w.show_checks(w.verify(reply.content, observations))

number,claim_id,the record holds,mark
7.2,IT.NET.USER.ZS:KEN:2010,7.2,verified
8.0,IT.NET.USER.ZS:RWA:2010,8,verified
16.6,IT.NET.USER.ZS:KEN:2015,16.5879,verified
12.5,IT.NET.USER.ZS:RWA:2015,12.456,verified
24.7,IT.NET.USER.ZS:KEN:2020,24.7228,verified
23.8,IT.NET.USER.ZS:RWA:2020,23.8,verified
35.0,IT.NET.USER.ZS:KEN:2024,34.9762,verified
31.7,IT.NET.USER.ZS:RWA:2024,31.7033,verified
1092,NY.GDP.PCAP.CD:KEN:2010,1091.52,verified
587,NY.GDP.PCAP.CD:RWA:2010,586.6,verified


Fourteen numbers are verified, none is flagged, and 21 are unmarked. The record for Kenya's 2024 internet use holds 34.9762; rounded to one decimal it is 35.0, so the comparison holds. The 21 unmarked numbers are the repeats and the computed values in the patterns. A computed value has no single record to check against, so the protocol leaves it unmarked, and the reader sees that it was not checked. A check by value alone, without tags, could not tell a source from a coincidence: 28, a difference the model computed, equals Rwanda's 2022 internet share of 28.0. Years are not figures and are skipped.

The chat view imitates `ClaimMark`, the component of the program's `@pcn-js/ui` package that draws the same marks in a web chatbot ([`worldbank/pcn`](https://github.com/worldbank/pcn)). The marks here come from the Python verifier.

## 2. A flagged number and an unmarked number

A flag means that the number does not match the record named in its tag. An unmarked number carries no tag, so there is nothing to compare, and no mark is drawn whether or not the number is right. The next cell replays a hand-made answer with one of each: Rwanda's internet use altered to 35.7 against the record's 31.7, and a figure of about 70 percent for mobile money that did not come from the tool.

In [5]:
w.use("nb06_claims_error")
flawed = model.invoke([SystemMessage(content=INSTRUCTIONS_CLAIMS), HumanMessage(content="DATA:\n" + json.dumps(results) + "\n\nQUESTION: " + w.MAIN_QUESTION)])
w.show_chat(w.MAIN_QUESTION, flawed.content, observations, results)
w.show_checks(w.verify(flawed.content, observations))

number,claim_id,the record holds,mark
35.0,IT.NET.USER.ZS:KEN:2024,34.9762,verified
35.7,IT.NET.USER.ZS:RWA:2024,31.7033; shown 35.7,flagged
"2,133",NY.GDP.PCAP.CD:KEN:2024,2133.46,verified
"1,060",NY.GDP.PCAP.CD:RWA:2024,1059.94,verified
70,no tag,none,unmarked


The verifier flags 35.7 with a ⚠, and the tooltip shows the record, the value returned, and the failed comparison. The 70 carries no mark. The absence is deliberate. The protocol states the principle as: *trust is earned only by proof, and the absence of a verification mark communicates uncertainty.* The check says nothing about whether the record is the right one for the question, whether the method behind it is sound, or whether the sentence around the number interprets it correctly. Those judgements remain with the reader.

## 3. Two configurations of the application

A check on one answer does not measure the application. The rest of the notebook compares two configurations on the same questions:

| Configuration | What it uses | Built in |
|---|---|---|
| **A · retrieval only** | The search, then the model writes from the passages. No tools, no tags. | Notebook 03 |
| **B · tools and claims** | The agent with both tools, the claim instructions above, and the verifier. | Notebooks 04 and 05, and this one |

The next cell defines each as a function of a question that returns the answer, the retrieved passages, and the tool observations. `app_rag` returns no observations, because nothing in A can be verified. `app_tools` reads the observations from the agent's calls to `get_indicator` with `w.tool_results`, and also retrieves passages, for the retrieval column of the table.

In [6]:
INSTRUCTIONS_RAG = """You are a research assistant for development data.
Answer only from the supplied passages and cite them in the square-bracket form given. If they do not contain what is needed, say so."""
rag_prompt = ChatPromptTemplate.from_messages([("system", INSTRUCTIONS_RAG), ("human", "PASSAGES:\n{context}\n\nQUESTION: {question}")])
rag_chain = rag_prompt | model
agent = create_agent(model, tools, system_prompt=INSTRUCTIONS_CLAIMS)


def app_rag(question, scenario):
    hits = store.similarity_search(question, k=4)
    w.use(scenario)
    reply = rag_chain.invoke({"context": w.format_context(hits), "question": question})
    return reply.content, hits, {}


def app_tools(question, scenario):
    w.use(scenario)
    result = agent.invoke({"messages": [HumanMessage(content=question)]}, config={"recursion_limit": 10})
    obs = w.observations_from_results(w.tool_results(result, "get_indicator"))
    hits = store.similarity_search(question, k=4)  # for the retrieval column; the agent may or may not have searched
    return w.final_text(result), hits, obs

## 4. The evaluation suite

A suite is a fixed list of questions, each with the documents and numbers a correct answer uses and the behaviour it should show. The next cell lists the eight questions of the session, one per kind, from `assets/questions.json`.

In [7]:
questions = w.load_questions()
w.show_table(["id", "kind", "question", "what a good answer does"], [(q["id"], q["category"], q["question"], q["behaviour"]) for q in questions])

id,kind,question,what a good answer does
q1,main,"How have internet access and economic development evolved in Rwanda and Kenya? Compare the recent available data, summarize the main patterns, and cite the sources.","Reports the latest year for each indicator and country, names the indicators and the source, and compares levels and trends."
q2,factual,What share of Rwanda's population used the internet in 2023?,Gives 28.8 percent with the indicator name and the source.
q3,ambiguous,How connected is Kenya?,"Asks which measure is meant, or states the indicator it chose before answering, and distinguishes internet use from mobile subscriptions."
q4,missing-data,What share of Kenya's population used the internet in 2025?,"States that no 2025 observation is available and gives the latest available year, 2024, at 35.0 percent."
q5,out-of-scope,How have Kenya's coffee export earnings changed in recent years?,States that the available documents describe the sector without figures and that the data tools hold no coffee-export indicator; gives no number.
q6,factual,Which of the two countries had faster GDP growth in 2023?,"Rwanda, 8.6 percent against 5.7 percent, with the source."
q7,definition,"How is the internet-users indicator defined, and what should be kept in mind when comparing years?","Gives the three-month, any-device definition from the ITU and at least one caveat on comparability."
q8,outdated-evidence,What are the latest available figures for internet use in Kenya and Rwanda?,"Gives the 2024 values, 35.0 and 31.7 percent. Quoting 2015 values from the 2016 summary is a failure."


The next cell runs both configurations through the eight questions. For each question it fills four columns: retrieval, whether an expected document is among the four passages (`w.retrieval_hit`); numbers, how many of the expected numbers appear in the answer (`w.numbers_present`); cited, whether the answer cites a retrieved passage or names the source (`w.grounded`); and the verifier's counts of verified, flagged, and unmarked numbers, for B only.

In [8]:
def evaluate(app, prefix):
    rows = []
    for qd in questions:
        answer, hits, obs = app(qd["question"], f"{prefix}_{qd['id']}")
        found, total = w.numbers_present(answer, qd["expected_numbers"])
        cited = w.grounded(answer, hits) or "WDI" in answer or "World Development Indicators" in answer
        if obs:
            checks = w.verify(answer, obs)
            v, f, u = (sum(c.status == s for c in checks) for s in ("verified", "flagged", "unmarked"))
            ver = f"{v} / {f} / {u}"
        else:
            ver = "not applicable" if prefix.endswith("rag") else "no tool data used"
        rows.append(w.EvalRow(qd["id"], qd["category"], "hit" if w.retrieval_hit(hits, qd["expected_documents"]) else "miss", f"{found}/{total}" if total else "none expected", "yes" if cited else "no", ver))
    return rows


print("A · retrieval only\n")
w.show_eval(evaluate(app_rag, "nb06_rag"))
print("\nB · tools and claims\n")
w.show_eval(evaluate(app_tools, "nb06_tools"))

A · retrieval only



id,category,retrieval,numbers,cited,verified / flagged / unmarked
q1,main,hit,2/4,yes,not applicable
q2,factual,hit,1/1,yes,not applicable
q3,ambiguous,hit,none expected,yes,not applicable
q4,missing-data,hit,none expected,yes,not applicable
q5,out-of-scope,hit,none expected,yes,not applicable
q6,factual,hit,2/2,yes,not applicable
q7,definition,hit,none expected,yes,not applicable
q8,outdated-evidence,hit,2/2,yes,not applicable



B · tools and claims



id,category,retrieval,numbers,cited,verified / flagged / unmarked
q1,main,hit,4/4,yes,16 / 0 / 18
q2,factual,hit,1/1,yes,1 / 0 / 0
q3,ambiguous,hit,none expected,yes,4 / 0 / 2
q4,missing-data,hit,none expected,yes,2 / 0 / 0
q5,out-of-scope,hit,none expected,yes,no tool data used
q6,factual,hit,0/2,yes,no tool data used
q7,definition,hit,none expected,yes,no tool data used
q8,outdated-evidence,hit,2/2,yes,5 / 0 / 0


## 5. Reading the table

- **q1, the main question.** A finds two of the four numbers: at `k = 4` the economy chunks fall below the cut-off, as notebook 03 showed. B finds all four, with 16 numbers verified and 18 unmarked repeats and computed values.
- **q6, which country grew faster in 2023.** A answers from the passages: Rwanda, 8.6 against 5.7 percent. B asks which two countries are meant and calls no tool, so it finds none of the two numbers. The question names no country. A had the passages, which do; the agent had nothing to infer the countries from.
- **q3, the ambiguous question.** B names three readings, fetches two series, and reports both with tags: four numbers verified. A answers internet use only and says the passages hold nothing on subscriptions.
- **q8, outdated evidence.** Both find the 2024 figures. At `k = 4` A retrieves both briefs and sets the 2016 summary aside.
- **The retrieval column reads hit and the cited column reads yes on every row of both tables.** Neither column would have revealed the two failures.

Each configuration fails one question that the other passes, and the table shows which. A table like this is rerun after every change to the instructions, the tools, the store, or the model. For suites at scale, with a model as a judge for the behaviours a rule cannot check, see the program's evaluation cookbook; libraries such as Ragas and DeepEval provide those judges.

## 6. What the suite does not cover

- Whether the application chose the right indicator for the question. A verified number from the wrong series is still verified.
- Whether the interpretation around the numbers is right.
- Behaviour on questions that nobody wrote down. Each failure found in use is added to the suite as a question.

## Key points

- A claim tag names the record a number comes from and the rule for comparing them. The verifier compares the two and marks the number verified or flagged. A number without a tag is unmarked.
- An unmarked number is neither right nor wrong; it is unchecked, and the reader is told so.
- A check on one answer is not a measure of the application. Evaluation runs fixed questions with known evidence through each configuration and compares the tables.
- In the session's suite, retrieval only fails the main question and tools with claims fails the question that names no country. The retrieval and citation columns alone would have passed both.

## Exercise

All of these run without a key, on the answer of section 1 (`reply`).

1. Change one number inside a tag: `tampered = reply.content.replace(">35.0<", ">35.7<")`, then `w.show_checks(w.verify(tampered, observations))`. Which mark changes, and what does the record hold?
2. Remove all the tags: `plain = w.strip_claims(reply.content)`, then `w.show_checks(w.verify(plain, observations))`. What happens to every number, and why is the result not a "verified" one?
3. Change the policy of one tag from rounded to exact: `strict = reply.content.replace('policy="rounded" decimals="1"', 'policy="exact"', 1)`, then verify it. Why does 35.0 stop matching a record of 34.9762, and which policy suits a published percentage?
4. Look at the eight questions of section 4. Which kind of question would be worth adding next, and what would a correct answer to it do?

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows. See also the [paper](https://arxiv.org/abs/2509.06902) and the [`pcn` repository](https://github.com/worldbank/pcn).
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.

**Next:** [07 · Lab: chat with your documents](https://colab.research.google.com/github/worldbank/ai4data/blob/main/training/session-01-building-ai-applications/notebooks/07-lab-chat-with-your-documents.ipynb)